# Asset Category — Batch Prediction on Spark Data

Runs the MLP classifier (`asset_category_inference.py`) over a Spark DataFrame
of assets and writes predictions back as new columns.

**Prerequisites** — upload these files to a Databricks Workspace path
(e.g. `/Workspace/Users/<you>/model_artifacts/Jaya_POC/CTEM/`):
- `asset_category_inference.py`
- `model_artifacts/*.pkl`  (10 pickle files)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Update these paths to match your Databricks workspace layout.

# Where asset_category_inference.py lives
INFERENCE_MODULE_DIR = "/Workspace/Users/jramajayam@infoblox.com/model_artifacts/Jaya_POC/CTEM"

# Where the .pkl artifacts live (model_artifacts subfolder)
ARTIFACTS_DIR = f"{INFERENCE_MODULE_DIR}/model_artifacts"

# Spark table / view containing asset data
ASSET_TABLE = "your_catalog.your_schema.your_asset_table"

# Batch size for pandas UDF processing (rows per partition)
BATCH_SIZE = 5000

In [ ]:
# ── Imports & model loading ───────────────────────────────────────────────────
import sys
import json
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, FloatType,
)

# Add inference module to path
if INFERENCE_MODULE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_MODULE_DIR)

from asset_category_inference import AssetCategoryClassifier

# Load model once on the driver (will be broadcast to workers)
print("Loading model on driver...")
clf = AssetCategoryClassifier(artifacts_dir=ARTIFACTS_DIR)
print(f"✅ Model loaded — {len(clf.categories)} categories")

In [ ]:
# ── Load asset data ───────────────────────────────────────────────────────────
# Option 1: From a table
# df_assets = spark.table(ASSET_TABLE)

# Option 2: From a CSV in DBFS / Volumes
# df_assets = spark.read.csv("/Volumes/your_catalog/your_schema/your_volume/asset_2.csv",
#                            header=True, inferSchema=True)

# Option 3: From a Delta table
# df_assets = spark.read.format("delta").load("/path/to/delta/table")

# --- Uncomment ONE of the above, or adjust to your data source ---
df_assets = spark.table(ASSET_TABLE)

# Select only the columns the model needs + asset_id for joining back
df_input = df_assets.select(
    "asset_id",
    "domain_name",
    "ports",
    "banners",
    "technologies",
    "passive_dns_query_count",
)

total = df_input.count()
print(f"Assets to process: {total:,}")

## Approach: Pandas UDF (vectorised)

We use `mapInPandas` to stream partitions through the classifier in chunks.
The model is **broadcast** to every executor, avoiding repeated loads.

In [ ]:
# ── Broadcast configuration to all executors ──────────────────────────────────
# NOTE: We do NOT pickle the classifier itself because sklearn's TfidfVectorizer
# loses its fitted idf_ attribute during pickle serialization.
# Instead, we broadcast the paths and load the model fresh on each executor.

bc_inference_dir = spark.sparkContext.broadcast(INFERENCE_MODULE_DIR)
bc_artifacts_dir = spark.sparkContext.broadcast(ARTIFACTS_DIR)

print(f"✅ Broadcast paths to all executors")
print(f"   Inference module: {INFERENCE_MODULE_DIR}")
print(f"   Artifacts dir:    {ARTIFACTS_DIR}")

In [ ]:
# ── Helpers: parse Spark column values into model input dicts ─────────────────

def _is_null(val):
    """Check if a value is None / NaN / NaT (numpy-safe, avoids truth-of-array)."""
    if val is None:
        return True
    # numpy arrays are never considered "null" – let the caller handle them
    if isinstance(val, np.ndarray):
        return False
    # lists/tuples are never null
    if isinstance(val, (list, tuple)):
        return False
    try:
        return bool(pd.isna(val))
    except (ValueError, TypeError):
        # pd.isna raises ValueError on some exotic types
        return False


def _parse_ports(val):
    """Parse ports from various formats: array, postgres {}, numpy array, or JSON."""
    if _is_null(val):
        return []
    # numpy array → list
    if isinstance(val, np.ndarray):
        return [str(p) for p in val.tolist()]
    if isinstance(val, (list, tuple)):
        return [str(p) for p in val]
    s = str(val).strip()
    if s in ('', '{}', 'NULL', 'None', '[]'):
        return []
    # Postgres array: {80,443}
    if s.startswith('{') and s.endswith('}'):
        return [x.strip().strip('"') for x in s[1:-1].split(',') if x.strip()]
    # JSON array: [80, 443]
    try:
        items = json.loads(s)
        if isinstance(items, list):
            return [str(p) for p in items]
    except (json.JSONDecodeError, TypeError):
        pass
    # Comma-separated fallback
    return [x.strip() for x in s.split(',') if x.strip()]


def _parse_technologies(val):
    """Parse technologies JSON: [{"name":"nginx"}] → ['nginx']."""
    if _is_null(val):
        return []
    if isinstance(val, np.ndarray):
        val = val.tolist()
    if isinstance(val, (list, tuple)):
        names = []
        for item in val:
            if isinstance(item, dict):
                n = item.get('name', '')
            elif hasattr(item, 'name'):
                n = item.name
            else:
                n = str(item)
            if n:
                names.append(n)
        return names
    s = str(val).strip()
    if s in ('', '{}', 'NULL', 'None', '[]'):
        return []
    try:
        items = json.loads(s)
        if isinstance(items, list):
            return [d.get('name', '') for d in items
                    if isinstance(d, dict) and d.get('name')]
    except (json.JSONDecodeError, TypeError):
        pass
    return []


def _safe_str(val, default=''):
    """Convert to str, handling None / NaN / arrays safely."""
    if _is_null(val):
        return default
    return str(val)


def _safe_int(val, default=0):
    """Convert to int, handling None / NaN / arrays safely."""
    if _is_null(val):
        return default
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default


def _row_to_asset(row):
    """Convert a pandas row to the dict format expected by the classifier."""
    return {
        'ports': _parse_ports(row.get('ports')),
        'banners': _safe_str(row.get('banners')),
        'domain_name': _safe_str(row.get('domain_name')),
        'technologies': _parse_technologies(row.get('technologies')),
        'passive_dns_query_count': _safe_int(row.get('passive_dns_query_count')),
    }

In [ ]:
# ── mapInPandas function ──────────────────────────────────────────────────────

# Cache the classifier at the executor level (loaded once per Python worker)
_executor_clf_cache = {}

def predict_partition(iterator):
    """Process partitions of pandas DataFrames through the MLP classifier."""
    import sys
    
    # Load model once per executor (not per partition)
    if 'clf' not in _executor_clf_cache:
        inference_dir = bc_inference_dir.value
        artifacts_dir = bc_artifacts_dir.value
        
        # Ensure inference module is importable
        if inference_dir not in sys.path:
            sys.path.insert(0, inference_dir)
        
        # Import and instantiate fresh — avoids pickle issues with TfidfVectorizer
        from asset_category_inference import AssetCategoryClassifier
        _executor_clf_cache['clf'] = AssetCategoryClassifier(artifacts_dir=artifacts_dir)
    
    _clf_local = _executor_clf_cache['clf']

    for pdf in iterator:
        if pdf.empty:
            yield pdf.assign(
                predicted_categories=pd.Series(dtype='object'),
                top_category=pd.Series(dtype='str'),
                top_probability=pd.Series(dtype='float'),
                all_probabilities=pd.Series(dtype='object'),
            )
            continue

        # Convert rows → asset dicts
        assets = [_row_to_asset(row) for _, row in pdf.iterrows()]

        # Run batch prediction
        results = _clf_local.predict_batch(assets)

        # Attach results
        pdf = pdf.copy()
        pdf['predicted_categories'] = [json.dumps(r['categories']) for r in results]
        pdf['top_category'] = [
            list(r['probabilities'].keys())[0] if r['probabilities'] else ''
            for r in results
        ]
        pdf['top_probability'] = [
            float(list(r['probabilities'].values())[0]) if r['probabilities'] else 0.0
            for r in results
        ]
        pdf['all_probabilities'] = [
            json.dumps(r['probabilities']) for r in results
        ]

        yield pdf

In [ ]:
# ── Define output schema & run predictions ────────────────────────────────────

output_schema = StructType(
    [StructField(c.name, c.dataType, c.nullable) for c in df_input.schema]
    + [
        StructField("predicted_categories", StringType(), True),
        StructField("top_category", StringType(), True),
        StructField("top_probability", FloatType(), True),
        StructField("all_probabilities", StringType(), True),
    ]
)

num_partitions = max(1, total // BATCH_SIZE)
print(f"Processing {total:,} assets across {num_partitions} partitions (~{BATCH_SIZE} rows each)")

df_predicted = (
    df_input
    .repartition(num_partitions)
    .mapInPandas(predict_partition, schema=output_schema)
)

# Cache for downstream use
df_predicted.cache()

# Trigger computation
predicted_count = df_predicted.count()
print(f"✅ Predictions complete: {predicted_count:,} assets processed")

In [ ]:
# ── Preview results ──────────────────────────────────────────────────────────
display(
    df_predicted
    .select("asset_id", "domain_name", "predicted_categories",
            "top_category", "top_probability")
    .limit(20)
)

In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────────
from pyspark.sql.functions import explode, from_json, col

print("Category distribution (top 20):")

df_stats = (
    df_predicted
    .withColumn("cats_array",
                from_json(col("predicted_categories"), ArrayType(StringType())))
    .withColumn("cat", explode("cats_array"))
)

display(
    df_stats
    .groupBy("cat")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

In [ ]:
# ── (Optional) Save predictions ──────────────────────────────────────────────

# Option 1: Delta table
# df_predicted.write.format("delta").mode("overwrite") \
#     .saveAsTable("your_catalog.your_schema.asset_predictions")

# Option 2: CSV
# df_predicted.toPandas().to_csv(
#     "/dbfs/tmp/asset_predictions.csv", index=False
# )

# Option 3: Join back to original table
# df_final = df_assets.join(
#     df_predicted.select("asset_id", "predicted_categories",
#                         "top_category", "top_probability"),
#     on="asset_id", how="left"
# )

print("Done! Uncomment a save option above to persist results.")